In [1]:
# Intialization
import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession

In [4]:


spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Demo'). \
    master('local[*]'). \
    getOrCreate()

In [5]:
sc = spark.sparkContext

In [6]:
from pyspark.sql.types import *
import pyspark.sql.functions as F

schema=StructType(
    [
        StructField('order_id', StringType(), True),
        StructField('order_timestamp', TimestampType(), True),
        StructField('customer_id', StringType(), True),
        StructField('city', StringType(), True),
        StructField('category', StringType(), True),
        StructField('product_id', StringType(), True),
        StructField('quantity', IntegerType(), True),
        StructField('unit_price', DoubleType(), True),
        StructField('discount_pct', DoubleType(), True),
        StructField('payment_mode', StringType(), True),
        StructField('order_status', StringType(), True)
             
    ]
)

In [7]:
df=spark.read.format('csv')\
            .option('header', True)\
            .schema(schema)\
            .load("file:///home/talentum/test-jupyter/use-case1--spark/data/ecomorders_sample.csv")

df.show(truncate=False)

+---------+-------------------+-----------+---------+--------------+----------+--------+----------+------------+------------+------------+
|order_id |order_timestamp    |customer_id|city     |category      |product_id|quantity|unit_price|discount_pct|payment_mode|order_status|
+---------+-------------------+-----------+---------+--------------+----------+--------+----------+------------+------------+------------+
|ORD000301|2025-09-05 22:01:27|CUST0025   |Delhi    |Books         |P0210     |2       |201.26    |5.0         |COD         |DELIVERED   |
|ORD000379|2025-12-02 07:06:18|CUST0020   |Pune     |Fashion       |P0290     |2       |2527.91   |0.0         |Wallet      |DELIVERED   |
|ORD000585|2025-01-19 15:22:26|CUST0089   |Ahmedabad|Electronics   |P0267     |1       |10806.18  |5.0         |COD         |DELIVERED   |
|ORD000354|2025-01-24 20:53:43|CUST0023   |Pune     |Grocery       |P0003     |1       |916.83    |25.0        |Wallet      |DELIVERED   |
|ORD000002|2025-02-17 06:14

In [8]:
df.filter("customer_id IS NULL").count()

28

Remove duplicate order_id rows, drop rows where customer_id is null and replace null discount_pct with 0. Print the record count before and after each cleaning step.

In [9]:
print(df.count())

646


In [10]:
# Step 2: Remove duplicate order_id rows (keep first occurrence)
df_no_duplicates = df.dropDuplicates(["order_id"])
print("Record count after removing duplicates:", df_no_duplicates.count())

# Step 3: Drop rows where customer_id is null
df_no_null_customer = df_no_duplicates.filter(F.col("customer_id").isNotNull())
print("Record count after dropping null customer_id:", df_no_null_customer.count())

# Step 4: Replace null values in discount_pct with 0.0
df_cleaned = df_no_null_customer.fillna({"discount_pct": 0.0})
print("Record count after replacing null discount_pct with 0:", df_cleaned.count())
print("=" * 55)

# Optional: Show final cleaned data
df_cleaned.show(5, truncate=False)

Record count after removing duplicates: 606
Record count after dropping null customer_id: 581
Record count after replacing null discount_pct with 0: 581
+---------+-------------------+-----------+-------+--------------+----------+--------+----------+------------+------------+------------+
|order_id |order_timestamp    |customer_id|city   |category      |product_id|quantity|unit_price|discount_pct|payment_mode|order_status|
+---------+-------------------+-----------+-------+--------------+----------+--------+----------+------------+------------+------------+
|ORD000055|2025-12-17 19:51:07|CUST0007   |Mumbai |Toys          |P0007     |1       |2615.32   |0.0         |Net Banking |DELIVERED   |
|ORD000163|2025-12-19 01:10:28|CUST0076   |Kolkata|Grocery       |P0368     |1       |563.07    |15.0        |Debit Card  |DELIVERED   |
|ORD000249|2025-06-29 17:17:38|CUST0029   |Delhi  |Toys          |P0070     |2       |1312.67   |0.0         |Net Banking |CANCELLED   |
|ORD000332|2025-06-09 04:

In [15]:
from pyspark.sql.window import Window

# Compute net_amount on the base DataFrame (assuming input DataFrame is named 'df')
df_with_net = df_cleaned.withColumn(
    "net_amount",
    F.round(F.col("quantity") * F.col("unit_price") * (1 - F.col("discount_pct") / 100.0), 2)
)

# Filter for DELIVERED orders to reuse across tasks
delivered_df = df_with_net.filter(F.col("order_status") == "DELIVERED")

In [16]:
monthly_category_rev = (
    delivered_df
    .withColumn("order_month", F.date_format(F.col("order_timestamp"), "yyyy-MM"))
    .groupBy("order_month", "category")
    .agg(F.round(F.sum("net_amount"), 2).alias("total_net_amount"))
    .orderBy("order_month", "category")
)

monthly_category_rev.show(truncate=False)

+-----------+--------------+----------------+
|order_month|category      |total_net_amount|
+-----------+--------------+----------------+
|2025-01    |Beauty        |62145.67        |
|2025-01    |Books         |15510.67        |
|2025-01    |Electronics   |588778.4        |
|2025-01    |Fashion       |105400.3        |
|2025-01    |Grocery       |28314.5         |
|2025-01    |Home & Kitchen|170529.82       |
|2025-01    |Sports        |102687.27       |
|2025-01    |Toys          |57116.92        |
|2025-02    |Beauty        |3240.51         |
|2025-02    |Books         |3388.19         |
|2025-02    |Fashion       |24935.71        |
|2025-02    |Grocery       |6831.03         |
|2025-02    |Home & Kitchen|30210.97        |
|2025-02    |Sports        |103439.71       |
|2025-02    |Toys          |10341.13        |
|2025-03    |Beauty        |5129.75         |
|2025-03    |Books         |6374.1          |
|2025-03    |Electronics   |65252.56        |
|2025-03    |Fashion       |25107.

In [17]:
# Define window partitioned by month, ordered by highest revenue
month_window = Window.partitionBy("order_month").orderBy(F.col("total_net_amount").desc())

top_category_per_month = (
    monthly_category_rev
    .withColumn("rnk", F.row_number().over(month_window)) # Use rank() or dense_rank() if you want to keep ties
    .filter(F.col("rnk") == 1)
    .drop("rnk")
    .orderBy("order_month")
)

top_category_per_month.show(truncate=False)

+-----------+--------------+----------------+
|order_month|category      |total_net_amount|
+-----------+--------------+----------------+
|2025-01    |Electronics   |588778.4        |
|2025-02    |Sports        |103439.71       |
|2025-03    |Sports        |1360098.49      |
|2025-04    |Electronics   |284882.89       |
|2025-05    |Electronics   |127941.35       |
|2025-06    |Home & Kitchen|1496245.34      |
|2025-07    |Electronics   |324107.74       |
|2025-08    |Electronics   |138923.5        |
|2025-09    |Home & Kitchen|42584.78        |
|2025-10    |Sports        |64068.74        |
|2025-11    |Electronics   |80155.06        |
|2025-12    |Electronics   |173675.4        |
+-----------+--------------+----------------+



In [18]:
# Step 1: Compute total spend per customer per city
customer_city_spend = (
    delivered_df  # Change to df_with_net if all order statuses should be included
    .groupBy("city", "customer_id")
    .agg(F.round(F.sum("net_amount"), 2).alias("total_spend"))
)

# Step 2: Apply dense_rank over each city ordered by total_spend descending
city_window = Window.partitionBy("city").orderBy(F.col("total_spend").desc())

top_3_customers_per_city = (
    customer_city_spend
    .withColumn("rank", F.dense_rank().over(city_window))
    .filter(F.col("rank") <= 3)
    .select("city", "customer_id", "total_spend", "rank")
    .orderBy("city", "rank")
)

top_3_customers_per_city.show(truncate=False)

+---------+-----------+-----------+----+
|city     |customer_id|total_spend|rank|
+---------+-----------+-----------+----+
|Ahmedabad|CUST0091   |157195.62  |1   |
|Ahmedabad|CUST0092   |87389.9    |2   |
|Ahmedabad|CUST0088   |65344.38   |3   |
|Bengaluru|CUST0048   |394271.1   |1   |
|Bengaluru|CUST0044   |294720.22  |2   |
|Bengaluru|CUST0041   |231025.15  |3   |
|Chennai  |CUST0058   |284705.27  |1   |
|Chennai  |CUST0056   |252441.21  |2   |
|Chennai  |CUST0051   |87823.87   |3   |
|Delhi    |CUST0028   |264211.26  |1   |
|Delhi    |CUST0026   |143590.54  |2   |
|Delhi    |CUST0031   |101889.97  |3   |
|Hyderabad|CUST0061   |319263.27  |1   |
|Hyderabad|CUST0062   |136897.1   |2   |
|Hyderabad|CUST0066   |69861.58   |3   |
|Kolkata  |CUST0084   |300443.16  |1   |
|Kolkata  |CUST0083   |231414.33  |2   |
|Kolkata  |CUST0081   |128112.09  |3   |
|Mumbai   |CUST0003   |326304.02  |1   |
|Mumbai   |CUST0008   |59490.38   |2   |
+---------+-----------+-----------+----+
only showing top

In [19]:
# Step 1: Aggregate daily revenue for DELIVERED orders
daily_rev = (
    delivered_df
    .withColumn("order_date", F.to_date(F.col("order_timestamp")))
    .groupBy("order_date")
    .agg(F.round(F.sum("net_amount"), 2).alias("daily_revenue"))
)

# Step 2: Define the 7-day rolling window frame
rolling_7d_window = (
    Window.orderBy("order_date")
    .rowsBetween(-6, 0)
)

# Step 3: Compute the 7-day moving average
daily_rev_ma = daily_rev.withColumn(
    "moving_avg_7d",
    F.round(F.avg("daily_revenue").over(rolling_7d_window), 2)
)

daily_rev_ma.show(truncate=False)

# Step 4: Save to HDFS in Parquet format
daily_rev_ma.write.mode("overwrite").parquet("/user/UseCase1-Spark/daily_revenue")

+----------+-------------+-------------+
|order_date|daily_revenue|moving_avg_7d|
+----------+-------------+-------------+
|2025-01-01|73149.99     |73149.99     |
|2025-01-02|23234.05     |48192.02     |
|2025-01-03|32778.6      |43054.21     |
|2025-01-04|32867.71     |40507.59     |
|2025-01-05|75166.54     |47439.38     |
|2025-01-07|2459.84      |39942.79     |
|2025-01-08|67484.35     |43877.3      |
|2025-01-09|36884.72     |38696.54     |
|2025-01-10|56833.11     |43496.41     |
|2025-01-11|89889.41     |51655.1      |
|2025-01-12|63128.25     |55978.03     |
|2025-01-14|7936.8       |46373.78     |
|2025-01-15|142307.37    |66352.0      |
|2025-01-16|18087.83     |59295.36     |
|2025-01-17|93736.97     |67417.11     |
|2025-01-18|20085.18     |62167.4      |
|2025-01-19|42219.6      |55357.43     |
|2025-01-20|5542.34      |47130.87     |
|2025-01-21|3029.72      |46429.86     |
|2025-01-23|17555.23     |28608.12     |
+----------+-------------+-------------+
only showing top